# ANOMALY DETECTION IN LARGE-SCALE CLOUD SYSTEMS
[Paper](https://arxiv.org/abs/2411.09047)
- Large-scale anomaly detection dataset collected from IBM Cloud's Console over approximately 4.5 months (from January 22, 2024, to June 7, 2024.)
-  This high-dimensional dataset captures telemetry data from multiple data centers, specifically designed to aid researchers in developing and benchmarking anomaly detection methods in large-scale cloud environments.
- It contains 39,365 entries, each representing a 5-minute interval, with 117,448 features/attributes, as interval_start is used as the index.
- The dataset includes detailed information on request counts, HTTP response codes, and various aggregated statistics.
- The dataset also includes labeled anomaly events identified through IBM's internal monitoring tools, providing a comprehensive resource for real-world anomaly detection research and evaluation.

## File Descriptions
- `location_downtime.csv` - Details planned and unplanned downtimes for IBM Cloud data centers, including start and end times in ISO 8601 format.
- `unpivoted_data.parquet` - Contains raw telemetry data with 413 million+ rows, covering details like location, HTTP status codes, request types, and aggregated statistics (min, max, median response times).
  - `pivoted_data_all.parquet` - Pivoted version of the telemetry dataset with 39,365 rows and 117,449 columns, including aggregated statistics across multiple metrics and intervals.
- `anomaly_windows.csv` - Ground truth for anomalies, listing start and end times of recorded anomalies, categorized by source (Issue Tracker, Instant Messenger, Test Log).

demo/demo.[ipynb|html]: This demo file provides examples of how to access data in the Parquet files, available in Jupyter Notebook (.ipynb) and HTML (.html) formats, respectively.

IBM Cloud is a public cloud infrastructure with a global network of
over 60 data centers.  
The CONSOLE production software system is deployed across seven data centers worldwide, though not all instances are active at the same time; they are rotated based on operational requirements. Additional CONSOLE deployments are spread globally for testing and staging environments. The CONSOLE uses a microservices architecture, with each microservice generating millions of daily logs and telemetry records (a common challenge with large scale Cloud sys- tems).  
The sheer volume and variability of logs and telemetry, combined with the dynamic nature of containerized environments — characterized by transient workloads, volatile resource usage, and continuous scaling — necessitate advanced, context-aware anomaly detection techniques.

We collected telemetry data from the IBM Cloud CONSOLE, including logs and metrics generated by its microservices. Due to the large volume of logs and telemetry emitted by the microservices, we used a Publish/Subscribe (Pub/Sub) mechanism to efficiently manage the data collection. The microservices publish their logs through a Redis Pub/Sub system as part of this process.

In [ ]:
import os
import pandas as pd
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ibm_anomaly") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:

# Download Parquet files locally first
parquet_files = {
    "pivoted_data_all.parquet": "https://zenodo.org/records/14062900/files/pivoted_data_all.parquet?download=1",
    "unpivoted_data.parquet": "https://zenodo.org/records/14062900/files/unpivoted_data.parquet?download=1"
}

for filename, url in parquet_files.items():
    if not os.path.exists(filename):
        !wget -q $url -O $filename

# Load the data
anomaly_windows=pd.read_csv("https://zenodo.org/records/14062900/files/anomaly_windows.csv?download=1")
location_downtime=pd.read_csv("https://zenodo.org/records/14062900/files/location_downtime.csv?download=1")
pivoted_data_all=spark.read.parquet("pivoted_data_all.parquet")
unpivoted_data=spark.read.parquet("unpivoted_data.parquet")

In [ ]:
print('ANOMALY WINDOWS \n',anomaly_windows.shape)
anomaly_windows.head()

ANOMALY WINDOWS 
 (25, 4)


,number,anomaly_start,anomaly_end,anomaly_source
0,a1,2024-02-02 10:22:00-0500,2024-02-02 11:01:00-0500,2
1,a2,2024-02-07 05:38:00-0500,2024-02-07 09:13:00-0500,2
2,a3,2024-02-12 11:36:52-0500,2024-02-12 12:13:17-0500,3
3,a4,2024-02-12 11:41:08-0500,2024-02-12 12:12:49-0500,3
4,a5,2024-02-12 11:42:24-0500,2024-02-12 12:09:33-0500,3


In [ ]:
print('LOCATION DOWNTIME \n',location_downtime.shape)
location_downtime.head()

LOCATION DOWNTIME 
 (93, 3)


,location,downtime_start,downtime_end
0,datacenter3,2024-02-08 21:05:00+00:00,2024-02-08 21:25:00+00:00
1,datacenter3,2024-02-19 16:25:00+00:00,2024-02-19 16:50:00+00:00
2,datacenter3,2024-02-22 20:05:00+00:00,2024-02-22 22:40:00+00:00
3,datacenter3,2024-03-04 15:25:00+00:00,2024-03-04 15:50:00+00:00
4,datacenter3,2024-03-13 16:10:00+00:00,2024-03-13 16:35:00+00:00


In [ ]:
print('PIVOTED DATA \n')
# The dataset description indicates pivoted_data_all has 39,365 entries.
print('Count: 39365 (from dataset description to avoid potential OutOfMemoryError on count())')
print(f'Actual Rows (computed): {pivoted_data_all.count()}')
print(f'Columns: {len(pivoted_data_all.columns)}')

print('\nShowing first 5 rows with selected columns to avoid OutOfMemoryError:')
pivoted_data_all.select('interval_start', 'datacenter1_CLIENT_component10_GET_200_endpoint865_avg', 'datacenter1_CLIENT_component10_GET_200_endpoint865_max').show(5)

PIVOTED DATA 

Count: 39365 (from dataset description to avoid potential OutOfMemoryError on count())
Actual Rows (computed): 39365
Columns: 117449

Showing first 5 rows with selected columns to avoid OutOfMemoryError:
+--------------+------------------------------------------------------+------------------------------------------------------+
|interval_start|datacenter1_CLIENT_component10_GET_200_endpoint865_avg|datacenter1_CLIENT_component10_GET_200_endpoint865_max|
+--------------+------------------------------------------------------+------------------------------------------------------+
|    1705951500|                                                  NULL|                                                  NULL|
|    1705951800|                                                  NULL|                                                  NULL|
|    1705952100|                                                  NULL|                                                  NULL|
|    1705952400|   

Given the `StackOverflowError` when attempting to build an expression for all 117,449 columns, we will demonstrate the zero-counting logic on a *small sample of columns* to illustrate the concept without hitting the JVM stack limit.

In [ ]:
from pyspark.sql.functions import col, when, lit
from functools import reduce

# Select a small, representative sample of numerical columns
# For demonstration, let's pick 10 columns after 'interval_start'
sample_numerical_cols = pivoted_data_all.columns[1:11] # Adjust this range as needed

print(f"Demonstrating zero count on a sample of {len(sample_numerical_cols)} columns:")
for c in sample_numerical_cols:
    print(f"- {c}")

# Create the zero-count expression for the sample columns
sample_zero_count_expression = reduce(
    lambda expr, column_name: expr + when(col(column_name) == lit(0), lit(1)).otherwise(lit(0)),
    sample_numerical_cols,
    lit(0)
)

# Select only the interval_start, the sample columns, and the new zero count column
sampled_df_for_zero_counts = pivoted_data_all.select(
    ['interval_start'] + sample_numerical_cols
)

# Add the new 'num_zeros_in_sample_row' column to the sampled DataFrame
print('\nAttempting to add num_zeros_in_sample_row column for the selected sample...')
sampled_df_with_zero_counts = sampled_df_for_zero_counts.withColumn(
    'num_zeros_in_sample_row', sample_zero_count_expression
)

# Display the first few rows of the result
print('\nDisplaying first 5 rows with num_zeros_in_sample_row for the sample columns:')
sampled_df_with_zero_counts.count()

Demonstrating zero count on a sample of 10 columns:
- datacenter1_CLIENT_component10_GET_200_endpoint865_avg
- datacenter1_CLIENT_component10_GET_200_endpoint865_count
- datacenter1_CLIENT_component10_GET_200_endpoint865_kurtosis
- datacenter1_CLIENT_component10_GET_200_endpoint865_max
- datacenter1_CLIENT_component10_GET_200_endpoint865_median
- datacenter1_CLIENT_component10_GET_200_endpoint865_min
- datacenter1_CLIENT_component10_GET_200_endpoint865_skewness
- datacenter1_CLIENT_component10_GET_200_endpoint865_std
- datacenter1_CLIENT_component10_GET_400_endpoint865_avg
- datacenter1_CLIENT_component10_GET_400_endpoint865_count

Attempting to add num_zeros_in_sample_row column for the selected sample...

Displaying first 5 rows with num_zeros_in_sample_row for the sample columns:


39365

In [ ]:
print('\nDisplaying first 5 rows with just interval_start and num_zeros_in_sample_row to avoid OOM:')
sampled_df_with_zero_counts.select('interval_start', 'num_zeros_in_sample_row').show(5)


Displaying first 5 rows with just interval_start and num_zeros_in_sample_row to avoid OOM:
+--------------+-----------------------+
|interval_start|num_zeros_in_sample_row|
+--------------+-----------------------+
|    1705951500|                      0|
|    1705951800|                      0|
|    1705952100|                      0|
|    1705952400|                      0|
|    1705952700|                      0|
+--------------+-----------------------+
only showing top 5 rows


In [ ]:
print('UNPIVOTED DATA \n',unpivoted_data.count(),unpivoted_data.printSchema())
unpivoted_data.show(5)

root
 |-- interval_start: integer (nullable = true)
 |-- location: string (nullable = true)
 |-- kind: string (nullable = true)
 |-- host: string (nullable = true)
 |-- method: string (nullable = true)
 |-- statusCode: long (nullable = true)
 |-- endpoint: string (nullable = true)
 |-- aggregated_stats_name: string (nullable = true)
 |-- aggregated_stats_value: double (nullable = true)

UNPIVOTED DATA 
 413241248 None
+--------------+-----------+------+-----------+------+----------+-----------+---------------------+----------------------+
|interval_start|   location|  kind|       host|method|statusCode|   endpoint|aggregated_stats_name|aggregated_stats_value|
+--------------+-----------+------+-----------+------+----------+-----------+---------------------+----------------------+
|    1705951500|datacenter1|CLIENT|component41|   GET|       200|endpoint892|                  avg|             108022.47|
|    1705951500|datacenter1|CLIENT|component41|   GET|       200|endpoint892|         